# Semantic Benchmark Python - Rich Graph Pipeline Validation

This validation notebook is intentionally verbose: it checks the prepared dataset, the generated graph artifacts, timing/coverage metrics, skipped parse records, and representative Joern graph renders. It is meant to answer both "did the pipeline run?" and "do the produced graphs look plausible?".


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name and not (PROJECT_ROOT / "pipelines").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

CONFIG_ROOTS = {
    "project_root": str(PROJECT_ROOT),
    "data_root": os.environ.get("DATA_ROOT", os.path.join(os.path.dirname(str(PROJECT_ROOT)), "data")),
    "outputs_root": os.environ.get("OUTPUT_BASE_DIR", os.path.join(os.path.dirname(str(PROJECT_ROOT)), "outputs")),
}

from spectral_code.evaluation.notebook_helpers import (
    bcb_spec,
    semantic_spec,
    xglue_spec,
    artifact_status_dataframe,
    display_dataset_overview,
    display_pipeline_validation,
    display_code_graph_side_by_side_examples,
    graph_manifest_summary_dataframe,
    timing_stats_dataframe,
    skipped_graph_parse_dataframe,
    notebook_output_dir,
    configure_notebook_style,
)

configure_notebook_style()
sns.set_theme(style="whitegrid", context="notebook")


## Configuration Roots

All paths are derived dynamically from the project root, data root, and outputs root.


In [ ]:
spec = semantic_spec("Python")
GRAPH_TYPES = ['cfg', 'ddg', 'cpg']
PRIMARY_GRAPH_TYPE = "cpg"
ANALYSIS_OUTPUT_DIR = notebook_output_dir(spec, CONFIG_ROOTS)

pd.DataFrame([
    {"root": key, "path": value} for key, value in CONFIG_ROOTS.items()
] + [{"root": "analysis_output_dir", "path": ANALYSIS_OUTPUT_DIR}])


## Artifact Inventory

This table verifies all required dataset and pipeline artifacts and shows file sizes for concrete files.


In [ ]:
artifact_df = artifact_status_dataframe(spec)
display(artifact_df)


## Dataset Shape and Code Size Distributions

This section validates label balance and basic snippet/code-size characteristics before inspecting graphs.


In [ ]:
pair_df = display_dataset_overview(spec)


## Graph Manifests, Timing, Coverage, and Skipped Layers

This is the core pipeline validation block. It summarizes raw features, cleaned graph shards, spectral feature manifests, timing stats, DOT coverage, and skipped/empty graph layer logs.


In [ ]:
display_pipeline_validation(spec, graph_types=GRAPH_TYPES, sample_label=1)


## Three Random Code vs. Joern Graph Examples

These examples are sampled from the sub-dataset and rendered as source text beside the selected Joern graph layer.


In [ ]:
random_method_ids = display_code_graph_side_by_side_examples(
    spec,
    n_examples=3,
    graph_type=PRIMARY_GRAPH_TYPE,
    seed=7,
    max_code_lines=42,
    max_nodes=90,
)


## Machine-Readable Validation Tables

These compact tables make it easier to export or compare validation summaries across sub-datasets.


In [ ]:
manifest_summary_df = graph_manifest_summary_dataframe(spec)
timing_df = timing_stats_dataframe(spec)
skipped_df = skipped_graph_parse_dataframe(spec, limit=100)

display(manifest_summary_df)
display(timing_df)
display(skipped_df if not skipped_df.empty else pd.DataFrame([{"status": "no skipped graph parse records found"}]))
